In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# --- Load config and data ---
try:
    with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)
except:
    print("Yaml configuration file not found!")

boardgames = pd.read_csv(config['input_data']['file'])

# --- Top 10 lists ---
top10_ranking = boardgames[["boardgame", "avg_rating"]].nlargest(10, "avg_rating")
top10_overall = boardgames[["boardgame", "total_plays"]].nlargest(10, "total_plays")
top10_wishlisted = boardgames[["boardgame", "wishlisted"]].nlargest(10, "wishlisted")

# --- Count appearances across lists ---
all_games = (
    list(top10_ranking["boardgame"]) +
    list(top10_overall["boardgame"]) +
    list(top10_wishlisted["boardgame"])
)

game_counts = {}
for game in all_games:
    game_counts[game] = game_counts.get(game, 0) + 1

# --- Color function: navy if in >1 list, orange otherwise ---
def get_colors(df):
    return ["navy" if game_counts.get(game, 0) > 1 else "orange" for game in df["boardgame"]]

# --- Format number with dot as thousands separator ---
def format_number(value):
    return f"{value:,.0f}".replace(",", ".")

# --- Plot: Top 10 Rated ---
plt.figure(figsize=(8,6))
sns.barplot(x="avg_rating", y="boardgame", data=top10_ranking, palette=get_colors(top10_ranking))
for i, v in enumerate(top10_ranking["avg_rating"]):
    plt.text(v + 0.1, i, f"{v:.2f}", color="black", va="center")
plt.xlim(0, 10)
plt.xlabel("Rating 1 - 10")
plt.title("Top 10 Rated Games")
plt.show()

# --- Plot: Top 10 Most Played ---
plt.figure(figsize=(8,6))
sns.barplot(x="total_plays", y="boardgame", data=top10_overall, palette=get_colors(top10_overall))
for i, v in enumerate(top10_overall["total_plays"]):
    plt.text(v + 0.1, i, format_number(v), color="black", va="center")
plt.xlim(0, 900000)
plt.xlabel("Número de partidas jugadas")
plt.title("Top 10 Más Jugados")
plt.show()

# --- Plot: Top 10 Wishlisted ---
plt.figure(figsize=(8,6))
sns.barplot(x="wishlisted", y="boardgame", data=top10_wishlisted, palette=get_colors(top10_wishlisted))
for i, v in enumerate(top10_wishlisted["wishlisted"]):
    plt.text(v + 0.5, i, format_number(v), color="black", va="center")
plt.xlim(0, 30000)
plt.xlabel("Número en lista de deseos")
plt.title("Top 10 Más Deseados")
plt.show()


In [ ]:

# --- Correlation matrix for key metrics ---
cols = ["complexity", "amazon_price", "avg_rating", "total_plays"]
corr = boardgames[cols].corr()

# --- Pure orange heatmap ---
plt.figure(figsize=(6, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="Oranges",  # built-in sequential orange palette
    vmin=0, vmax=1,  # restrict to positive correlations
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title("Correlation Between Game Metrics", fontsize=14)
plt.tight_layout()
plt.show()